In [1]:
import numpy as np
import pandas as pd
import os
from joblib import Parallel, delayed
from copy import deepcopy
from dataclasses import dataclass, asdict
from tqdm import tqdm
from statistics import harmonic_mean
from typing import Dict, List, Any, Tuple
from metabci.brainda.datasets import Wang2016, BETA
from metabci.brainda.paradigms import SSVEP
from SUTLSSVEP import SUTLSSVEP
from metabci.brainda.algorithms.decomposition import SC_TRCA
from metabci.brainda.algorithms.decomposition import TtCCA
from metabci.brainda.algorithms.decomposition import TDCA
from metabci.brainda.algorithms.utils.model_selection import set_random_seeds
from metabci.brainda.algorithms.decomposition.base import generate_filterbank, generate_cca_references
from SQHAF import SQHAF
from iistlf import IISTLF
import torch
from EEGconformer import EEGConformer

In [2]:
#**************************************************
# Benchmark数据集读取处理
#**************************************************
Bench_dataset = Wang2016()
# 扩展到32个被试
Bench_subject_list = list(range(1, 36))

for s in Bench_subject_list:
    Bench_dataset.data_path(subject=s, path="/home/foam/metabci/mne_data")

events = Bench_dataset.events.keys()
freq_list = [str(Bench_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.64
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[Benchmark] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[Benchmark] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches

# ------------------------------------------------------

Bench_paradigm = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],  # 选择电极通道
    intervals=[(t_start, t_end)],
    events=freq_list,
    srate=sfreq
)

Bench_paradigm.register_raw_hook(raw_hook)

all_subjects = list(range(1, 36))


--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/y

In [3]:
# ============================================================
# 1) Utilities
# ============================================================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.5


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 Benchmark 数据；Bench_paradigm 和 Bench_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "benchmark":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = Bench_paradigm.get_data(
            Bench_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return []

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]

        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)

    # 使 target split、STC random split 和 source random selection 使用相同 seed
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None
    y_pred = model.predict(X_test, calib_X=calibration_X)

    acc = float(np.mean(y_pred == y_test))
    itr = compute_itr(
        acc,
        n_classes=run_cfg.n_classes,
        t_sec=run_cfg.decision_time,
    )

    return [{
        "dataset": run_cfg.dataset_name,
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": acc,
        "itr": itr,
        "n_test": int(len(y_test)),
        "stc_split_mode": model.stc_split_mode,
        "enable_harmonic_branch": bool(model.enable_harmonic_branch),
    }]


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)
    rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:
            seeds = [0] if n_calib == 0 else run_cfg.repeat_seeds

            for seed in seeds:
                out = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for result in out:
                    if result:
                        rows.extend(result)

    return pd.DataFrame(rows)


# ============================================================
# 6) Ablation variants
# ============================================================
# 当前默认列表对应：
#   - full：正式 time_ordered 方法
#   - stc_random：1.0 s STC random sensitivity
#   - w/o_harmonic：harmonic branch 关闭
#
# 如果要继续运行原来的其他消融，可将下面注释部分加入 variants。
variants = [
    VariantConfig(
        "full",
        {
            "stc_split_mode": "time_ordered",
            "enable_harmonic_branch": True,
        },
    ),
    VariantConfig(
        "stc_random",
        {
            "stc_split_mode": "random",
        },
    ),
    VariantConfig(
        "w/o_harmonic",
        {
            "enable_harmonic_branch": False,
        },
    ),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.5

base_model_kwargs = dict(
    n_sources=10,
    neighbor_radius=1,
    neighbor_decay=0.5,
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,
    target_alignment_mode="calibration",
    stc_split_mode="time_ordered",
    enable_harmonic_branch=True,
    enable_stage2=True,
    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)

run_cfg = RunConfig(
    dataset_name="Benchmark",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty. Please check data loading / model fitting.")


# ============================================================
# 8) Compact final summary
# ============================================================
df_summary = (
    df_raw.groupby(
        ["dataset", "variant", "n_calib"],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        n_runs=("acc", "count"),
    )
    .sort_values(["variant", "n_calib"])
)

print("\n========== Ablation Summary ==========")
print(
    df_summary[
        [
            "dataset",
            "variant",
            "n_calib",
            "acc_mean",
            "acc_std",
            "itr_mean",
            "itr_std",
            "n_runs",
        ]
    ].to_string(index=False)
)

--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/y

In [5]:
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """
    ITR bits/min
    acc: 准确率[0,1]
    n_classes: 类别数
    t_sec: 单次决策时间(秒)
    """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(dataset=dataset, subjects=[s], return_concat=True)
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


# =========================
# 全体被试 LOSO（不分组）
# =========================
all_subjects = list(range(1, 36))
n_classes = 40   # 按你的数据集类别数设置
T = 1.5    

print("\n========== LOSO on ALL subjects ==========")
print(f"n_classes={n_classes}, T={T}s")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)

all_results = []
accs, itrs = [], []

for target_sid in all_subjects:
    source_sids = [s for s in all_subjects if s != target_sid]
    source_data = {s: all_data[s] for s in source_sids}
    X_t, y_t = all_data[target_sid]

    model = SUTLSSVEP(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        n_harmonics=3,
        n_bands=5,
        top_m1=10,
        n_jobs=8
    )

    model.fit(source_data)
    y_pred = model.predict(X_t)

    acc = float(np.mean(y_pred == y_t))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=T))

    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

    all_results.append({
        "subject_id": target_sid,
        "subject": f"S{target_sid}",
        "acc": acc,
        "itr": itr,
        "n_trials": len(y_t)
    })

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


# =========================
# 最终汇总打印（仅打印，不保存csv）
# =========================
df = pd.DataFrame(all_results).sort_values("subject_id")

print("\n========== Subject-wise Results ==========")
print(df.to_string(index=False))

print("\n========== Overall Summary ==========")
summary_df = pd.DataFrame([{
    "acc_mean": df["acc"].mean(),
    "acc_std": df["acc"].std(),
    "itr_mean(bits/min)": df["itr"].mean(),
    "itr_std(bits/min)": df["itr"].std(),
    "n_subjects": df["subject_id"].nunique()
}])
print(summary_df.to_string(index=False))

print(f"\nOverall mean acc = {df['acc'].mean():.4f}")
print(f"Overall mean itr = {df['itr'].mean():.2f} bits/min")


========== LOSO on ALL subjects ==========
n_classes=40, T=1.5s
target S1: acc=0.8708, itr=163.36 bits/min
target S2: acc=0.8792, itr=166.06 bits/min
target S3: acc=0.9750, itr=200.85 bits/min
target S4: acc=0.9083, itr=175.82 bits/min
target S5: acc=0.9375, itr=186.17 bits/min
target S6: acc=0.7958, itr=140.51 bits/min
target S7: acc=0.8125, itr=145.39 bits/min
target S8: acc=0.4958, itr=66.29 bits/min
target S9: acc=0.7292, itr=121.91 bits/min
target S10: acc=0.8500, itr=156.77 bits/min
target S11: acc=0.2458, itr=21.25 bits/min
target S12: acc=0.9542, itr=192.45 bits/min
target S13: acc=0.7292, itr=121.91 bits/min
target S14: acc=0.8542, itr=158.07 bits/min
target S15: acc=0.7042, itr=115.29 bits/min
target S16: acc=0.5958, itr=88.50 bits/min
target S17: acc=0.6583, itr=103.59 bits/min
target S18: acc=0.7792, itr=135.72 bits/min
target S19: acc=0.2458, itr=21.25 bits/min
target S20: acc=0.8333, itr=151.64 bits/min
target S21: acc=0.6958, itr=113.12 bits/min
target S22: acc=0.9750, 

In [4]:
# IISTLF
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def pick_per_stimulus_trials(
    X,
    y,
    n_classes,
    n_calib_per_stimulus=2,
    random_state=42,
):
    """每个刺激类别抽取相同数量的校准 trial。"""
    rng = np.random.RandomState(random_state)
    selected = []

    for cls in range(n_classes):
        idx_cls = np.flatnonzero(y == cls)
        if len(idx_cls) < n_calib_per_stimulus:
            raise ValueError(
                f"目标被试类别 {cls} trial 不足: "
                f"{len(idx_cls)} < {n_calib_per_stimulus}"
            )
        selected.extend(
            rng.choice(
                idx_cls,
                size=n_calib_per_stimulus,
                replace=False,
            )
        )

    selected = np.sort(np.asarray(selected, dtype=int))
    mask = np.ones(len(y), dtype=bool)
    mask[selected] = False

    X_cal = X[selected]
    y_cal = y[selected]
    X_test = X[mask]
    y_test = y[mask]

    if len(X_test) == 0:
        raise ValueError("校准后没有剩余测试 trial。")
    if np.intersect1d(selected, np.flatnonzero(mask)).size:
        raise AssertionError("calibration/test selections overlap")
    if any(np.sum(y_cal == cls) != n_calib_per_stimulus for cls in range(n_classes)):
        raise AssertionError("每个刺激类别的校准 trial 数量不一致")

    return X_cal, y_cal, X_test, y_test


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（不分组）
# =========================
all_subjects = list(range(1, 36))
n_calib_per_stimulus = 1
random_state = 42

n_classes = len(freq_list)
t_sec = 1.5

print("\n========== LOSO on ALL subjects (IISTLF) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)
all_results = []
accs = []
itrs = []

for target_sid in all_subjects:
    X_t, y_t = all_data[target_sid]

    X_cal, y_cal, X_test, y_test = pick_per_stimulus_trials(
        X_t, y_t,
        n_classes=n_classes,
        n_calib_per_stimulus=n_calib_per_stimulus,
        random_state=random_state,
    )

    source_sids = [s for s in all_subjects if s != target_sid]
    src_accs = []

    for s_sid in source_sids:
        X_s, y_s = all_data[s_sid]

        model = IISTLF(
            srate=sfreq,
            freqs=[float(f) for f in freq_list],
            n_harmonics=3,
            n_subbands=5,          # 与 Table 1 披露的 IISTLF 子带数一致
        )
        model.fit(
            source_Xy=(X_s, y_s.astype(int)),
            target_calib_X=X_cal,
            target_calib_y=y_cal,
        )
        #   y_pred, scores = model.predict(X_test, return_scores=True)
        y_pred = model.predict(X_test)
        src_accs.append(float(np.mean(y_pred == y_test)))

    acc = float(np.mean(src_accs))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))

    accs.append(acc)
    itrs.append(itr)

    print(
        f"target S{target_sid}: acc={acc:.4f}, "
        f"itr={itr:.2f} bits/min "
        f"(avg over {len(source_sids)} sources)"
    )

    all_results.append(
        {
            "subject_id": target_sid,
            "subject": f"S{target_sid}",
            "acc": acc,
            "itr": itr,
        }
    )

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (IISTLF) ==========
n_classes=40, t_sec=1.5
target S1: acc=0.8490, itr=156.45 bits/min (avg over 34 sources)
target S2: acc=0.8916, itr=170.16 bits/min (avg over 34 sources)
target S3: acc=0.9788, itr=202.48 bits/min (avg over 34 sources)
target S4: acc=0.9451, itr=189.01 bits/min (avg over 34 sources)
target S5: acc=0.9700, itr=198.76 bits/min (avg over 34 sources)
target S6: acc=0.8457, itr=155.45 bits/min (avg over 34 sources)
target S7: acc=0.8088, itr=144.30 bits/min (avg over 34 sources)
target S8: acc=0.5466, itr=77.28 bits/min (avg over 34 sources)
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
target S9: acc=0.7256, itr=120.95 bits/min (avg over 34 sources)
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /

In [5]:
# **************************************************
# Benchmark数据集 eTRCA方法 - 全体数据LOSO实验（不分组，含ITR）
# **************************************************
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== eTRCA 固定参数配置 ==========
srate = 250
T = 1.5
n_harmonics = 3

# 滤波器组配置
wp = [(5, 90)]
ws = [(3, 92)]
filterbank = generate_filterbank(wp, ws, srate=srate, order=15, rp=0.5)

# 获取频率列表
events = Bench_dataset.events.keys()
freq_list = [Bench_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成参考信号
Yf = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA reference shape: {Yf.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")

# ========== 一次性获取全体数据 ==========
all_subjects = list(range(1, 36))
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)

print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    estimator = SC_TRCA(
        standard=False,
        ensemble=True,
        n_components=1
    )

    print("🔄 SC_TRCA训练中...")
    estimator.fit(X_train, y_train, Yf)
    print("✅ SC_TRCA训练完成")

    _, p_labels = estimator.predict(X_test)

    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA reference shape: (40, 6, 375)
📊 Number of classes: 40
📊 Decision time: 1.5s
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/y

In [4]:
# **************************************************
# Benchmark数据集 TtCCA - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== 固定参数 ==========
srate = 250
T = 1.5
n_harmonics = 3

events = Bench_dataset.events.keys()
freq_list = [Bench_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成CCA参考信号模板
cca_template = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA template shape: {cca_template.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")


# ========== 一次性获取全体数据（不分组） ==========
all_subjects = list(range(1, 36))
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)
print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO循环 ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化TtCCA模型
    estimator = TtCCA(
        n_components=1,
        n_jobs=-1
    )

    # 训练
    print("🔄 TtCCA训练中...")
    estimator.fit(X_train, y_train, cca_template, n_harmonics)
    print("✅ TtCCA训练完成")

    # 预测
    p_labels = estimator.predict(X_test)

    # 指标
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA template shape: (40, 6, 375)
📊 Number of classes: 40
📊 Decision time: 1.5s
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yi

In [5]:
# TDCA
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 参数区（可按需调整）
# =========================
n_harmonics = 3
window_length = 1.5      # 你的时间窗（秒）
ref_T = 1.4             # 关键修改：参考信号长度从0.5改为0.4，给padding留余量
padding_len = 1          # 关键修改：先用1更稳
n_components = 4

# ITR参数
n_classes = len(freq_list)
t_sec = window_length

print(f"[Benchmark] n_classes={n_classes}, t_sec={t_sec:.2f}s")

# =========================
# 参考信号（TDCA用）
# =========================
Yf = generate_cca_references(
    [float(f) for f in freq_list],
    srate=sfreq,
    T=ref_T,
    n_harmonics=n_harmonics
)
print(f"[Benchmark] CCA references shape: {Yf.shape}")

# =========================
# 一次性加载全体数据（LOSO）
# =========================
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,
    return_concat=True,
    n_jobs=None,
    verbose=False
)

all_subject_ids = np.unique(meta_all["subject"])
N = len(all_subject_ids)

print("\n" + "#" * 70)
print("# 🔬 TDCA LOSO on ALL subjects")
print("#" * 70)
print(f"Total subjects: {N}")
print(f"Subject IDs: {all_subject_ids}")
print(f"Data shape: {X_all.shape}")

# =========================
# LOSO循环
# =========================
acc_list = []
itr_list = []
results = []

for i, test_subject in enumerate(all_subject_ids):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all["subject"] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"Training samples: {len(train_ind)}")
    print(f"Testing samples: {len(test_ind)}")
    print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

    estimator = TDCA(
        padding_len=padding_len,
        n_components=n_components
    )

    # -------- 安全检查：避免 again 出现 length 报错 --------
    x_len = X_train.shape[-1]
    yf_len = Yf.shape[-1]   # 参考信号时间点
    if x_len <= (padding_len + yf_len):
        raise ValueError(
            f"Not enough time points for TDCA: "
            f"X_len={x_len}, padding_len={padding_len}, Yf_len={yf_len}, "
            f"need X_len > padding_len + Yf_len.\n"
            f"Try smaller ref_T, smaller padding_len, or longer data window."
        )

    p_labels = estimator.fit(
        X=X_train,
        y=y_train,
        Yf=Yf
    ).predict(X_test)

    acc_s = float(np.mean(p_labels == y_test))
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=t_sec))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    n_correct = int(np.sum(p_labels == y_test))
    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    results.append({
        "test_subject": int(test_subject),
        "accuracy": acc_s,
        "itr": itr_s,
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# =========================
# 汇总打印
# =========================
df_results = pd.DataFrame(results).sort_values("test_subject")

acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)

[Benchmark] n_classes=40, t_sec=1.50s
[Benchmark] CCA references shape: (40, 6, 350)
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /uploa

In [6]:
# EEGconformer
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（无目标校准、无目标微调）
# =========================
all_subjects = list(range(1, 36))
random_state = 42
n_classes = len(freq_list)
t_sec = 1.5

print("\n========== LOSO on ALL subjects (EEG-Conformer) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)
accs = []
itrs = []

for target_sid in all_subjects:
    X_test, y_test = all_data[target_sid]
    train_sids = [s for s in all_subjects if s != target_sid]
    X_train = np.concatenate([all_data[s][0] for s in train_sids], axis=0)
    y_train = np.concatenate([all_data[s][1] for s in train_sids], axis=0)

    model = EEGConformer(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        device="cuda" if torch.cuda.is_available() else "cpu",
        emb_size=40,
        depth=3,
        n_heads=4,
        batch_size=256,
        n_epochs=100,
        lr=3e-4,
        beta1=0.5,
        beta2=0.999,
        seed=random_state,
        use_amp=True,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test, batch_size=512)

    acc = float(np.mean(y_pred == y_test))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))
    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (EEG-Conformer) ==========
n_classes=40, t_sec=1.5
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
target S1: acc=0.6500, itr=101.52 bits/min
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/yijun/S25.mat.7z
--------ssssss, /

In [6]:
#**************************************************
# BETA数据集读取处理
#**************************************************
# BETA数据集，已经通过matlab的eegfit，进行了3-90HZ的带通滤波，故此处不再进行滤波处理
# BETA_wof数据集没有进行滤波处理，链接已经
# 在tsinghua.py把BETA_URL做以下修改即可使用未滤波的版本
# BETA_URL = "https://bci.med.tsinghua.edu.cn/upload/liubingchuan_BETA_wof/"


# 数据存储为一个四维张量 [channel, time point, block, condition]
#                    [   64,       750,      4,      40    ]

BETA_dataset = BETA()
subject_list = list(range(1, 71))  # 被试编号从1到70
for s in subject_list:
    BETA_dataset.data_path(subject=s, path="/home/foam/metabci/mne_data")  # 依次为每个被试设置路径

events = BETA_dataset.events.keys()
freq_list = [str(BETA_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率
BETA_freq_map = {i: freq for i, freq in enumerate(freq_list)}  # 标签到频率的映射
print(freq_list)  # 输出频率显示

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.64
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[BETA] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[BETA] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches


BETA_paradigm = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],
    intervals=[(t_start, t_end)],  # 提取1秒数据
    events=freq_list,
    srate=sfreq
)
BETA_paradigm.register_raw_hook(raw_hook)
all_subjects = list(range(1, 71))

--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchu

In [8]:
# ============================================================
# 1) Utilities
# ============================================================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.5


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 BETA 数据；BETA_paradigm 和 BETA_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "beta":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = BETA_paradigm.get_data(
            BETA_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return []

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]

        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)

    # 使 target split、STC random split 和 source random selection 使用相同 seed
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None
    y_pred = model.predict(X_test, calib_X=calibration_X)

    acc = float(np.mean(y_pred == y_test))
    itr = compute_itr(
        acc,
        n_classes=run_cfg.n_classes,
        t_sec=run_cfg.decision_time,
    )

    return [{
        "dataset": run_cfg.dataset_name,
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": acc,
        "itr": itr,
        "n_test": int(len(y_test)),
        "stc_split_mode": model.stc_split_mode,
        "enable_harmonic_branch": bool(model.enable_harmonic_branch),
    }]


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)
    rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:
            seeds = [0] if n_calib == 0 else run_cfg.repeat_seeds

            for seed in seeds:
                out = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for result in out:
                    if result:
                        rows.extend(result)

    return pd.DataFrame(rows)


# ============================================================
# 6) Ablation variants
# ============================================================
variants = [
    VariantConfig(
        "full",
        {
            "stc_split_mode": "time_ordered",
            "enable_harmonic_branch": True,
        },
    ),
    VariantConfig(
        "stc_random",
        {
            "stc_split_mode": "random",
        },
    ),
    VariantConfig(
        "w/o_harmonic",
        {
            "enable_harmonic_branch": False,
        },
    ),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.5

base_model_kwargs = dict(
    n_sources=10,
    neighbor_radius=1,
    neighbor_decay=0.5,
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,
    target_alignment_mode="calibration",
    stc_split_mode="time_ordered",
    enable_harmonic_branch=True,
    enable_stage2=True,
    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)

run_cfg = RunConfig(
    dataset_name="beta",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty. Please check data loading / model fitting.")


# ============================================================
# 8) Compact final summary
# ============================================================
df_summary = (
    df_raw.groupby(
        ["dataset", "variant", "n_calib"],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        n_runs=("acc", "count"),
    )
    .sort_values(["variant", "n_calib"])
)

print("\n========== Ablation Summary ==========")
print(
    df_summary[
        [
            "dataset",
            "variant",
            "n_calib",
            "acc_mean",
            "acc_std",
            "itr_mean",
            "itr_std",
            "n_runs",
        ]
    ].to_string(index=False)
)

--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchu

In [3]:
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """
    ITR bits/min
    acc: 准确率[0,1]
    n_classes: 类别数
    t_sec: 单次决策时间(秒)
    """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(dataset=dataset, subjects=[s], return_concat=True)
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


# =========================
# 全体被试 LOSO（不分组）
# =========================
# 你需要提前定义：
# all_subjects, sfreq, freq_list, BETA_paradigm, BETA_dataset, SUTLSSVEP

n_classes = 40   # 按你的数据集类别数设置
T = 1.5          # 单次决策时间(秒)，按实验窗口修改

print("\n========== LOSO on ALL subjects ==========")
print(f"n_classes={n_classes}, T={T}s")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)

all_results = []
accs, itrs = [], []

for target_sid in all_subjects:
    source_sids = [s for s in all_subjects if s != target_sid]
    source_data = {s: all_data[s] for s in source_sids}
    X_t, y_t = all_data[target_sid]

    model = SUTLSSVEP(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        n_harmonics=3,
        n_bands=5,
        top_m1=20,
        n_jobs=8
    )

    model.fit(source_data)
    y_pred = model.predict(X_t)

    acc = float(np.mean(y_pred == y_t))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=T))

    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

    all_results.append({
        "subject_id": target_sid,
        "subject": f"S{target_sid}",
        "acc": acc,
        "itr": itr,
        "n_trials": len(y_t)
    })

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


# =========================
# 最终汇总打印（仅打印，不保存csv）
# =========================
df = pd.DataFrame(all_results).sort_values("subject_id")

print("\n========== Subject-wise Results ==========")
print(df.to_string(index=False))

print("\n========== Overall Summary ==========")
summary_df = pd.DataFrame([{
    "acc_mean": df["acc"].mean(),
    "acc_std": df["acc"].std(),
    "itr_mean(bits/min)": df["itr"].mean(),
    "itr_std(bits/min)": df["itr"].std(),
    "n_subjects": df["subject_id"].nunique()
}])
print(summary_df.to_string(index=False))

print(f"\nOverall mean acc = {df['acc'].mean():.4f}")
print(f"Overall mean itr = {df['itr'].mean():.2f} bits/min")


========== LOSO on ALL subjects ==========
n_classes=40, T=1.5s
target S1: acc=0.8250, itr=149.12 bits/min
target S2: acc=0.9313, itr=183.89 bits/min
target S3: acc=0.8875, itr=168.80 bits/min
target S4: acc=0.6062, itr=90.94 bits/min
target S5: acc=0.7625, itr=131.03 bits/min
target S6: acc=0.6250, itr=95.42 bits/min
target S7: acc=0.5563, itr=79.43 bits/min
target S8: acc=0.6562, itr=103.07 bits/min
target S9: acc=0.6937, itr=112.58 bits/min
target S10: acc=0.5625, itr=80.83 bits/min
target S11: acc=0.3063, itr=30.66 bits/min
target S12: acc=0.8250, itr=149.12 bits/min
target S13: acc=0.6687, itr=106.20 bits/min
target S14: acc=0.7812, itr=136.31 bits/min
target S15: acc=0.7250, itr=120.80 bits/min
target S16: acc=0.6438, itr=99.98 bits/min
target S17: acc=0.4062, itr=48.37 bits/min
target S18: acc=0.9812, itr=203.54 bits/min
target S19: acc=0.7688, itr=132.78 bits/min
target S20: acc=0.5500, itr=78.03 bits/min
target S21: acc=0.8125, itr=145.39 bits/min
target S22: acc=0.6875, itr=

In [9]:
# IISTLF
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def pick_per_stimulus_trials(
    X,
    y,
    n_classes,
    n_calib_per_stimulus=2,
    random_state=42,
):
    """每个刺激类别抽取相同数量的校准 trial。"""
    rng = np.random.RandomState(random_state)
    selected = []

    for cls in range(n_classes):
        idx_cls = np.flatnonzero(y == cls)
        if len(idx_cls) < n_calib_per_stimulus:
            raise ValueError(
                f"目标被试类别 {cls} trial 不足: "
                f"{len(idx_cls)} < {n_calib_per_stimulus}"
            )
        selected.extend(
            rng.choice(
                idx_cls,
                size=n_calib_per_stimulus,
                replace=False,
            )
        )

    selected = np.sort(np.asarray(selected, dtype=int))
    mask = np.ones(len(y), dtype=bool)
    mask[selected] = False

    X_cal = X[selected]
    y_cal = y[selected]
    X_test = X[mask]
    y_test = y[mask]

    if len(X_test) == 0:
        raise ValueError("校准后没有剩余测试 trial。")
    if np.intersect1d(selected, np.flatnonzero(mask)).size:
        raise AssertionError("calibration/test selections overlap")
    if any(np.sum(y_cal == cls) != n_calib_per_stimulus for cls in range(n_classes)):
        raise AssertionError("每个刺激类别的校准 trial 数量不一致")

    return X_cal, y_cal, X_test, y_test


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（不分组）
# =========================
all_subjects = list(range(1, 71))
n_calib_per_stimulus = 1
random_state = 42

n_classes = len(freq_list)
t_sec = 1.5

print("\n========== LOSO on ALL subjects (IISTLF) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)
all_results = []
accs = []
itrs = []

for target_sid in all_subjects:
    X_t, y_t = all_data[target_sid]

    X_cal, y_cal, X_test, y_test = pick_per_stimulus_trials(
        X_t, y_t,
        n_classes=n_classes,
        n_calib_per_stimulus=n_calib_per_stimulus,
        random_state=random_state,
    )

    source_sids = [s for s in all_subjects if s != target_sid]
    src_accs = []

    for s_sid in source_sids:
        X_s, y_s = all_data[s_sid]

        model = IISTLF(
            srate=sfreq,
            freqs=[float(f) for f in freq_list],
            n_harmonics=3,
            n_subbands=5,          # 与 Table 1 披露的 IISTLF 子带数一致
        )
        model.fit(
            source_Xy=(X_s, y_s.astype(int)),
            target_calib_X=X_cal,
            target_calib_y=y_cal,
        )
        #   y_pred, scores = model.predict(X_test, return_scores=True)
        y_pred = model.predict(X_test)
        src_accs.append(float(np.mean(y_pred == y_test)))

    acc = float(np.mean(src_accs))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))

    accs.append(acc)
    itrs.append(itr)

    print(
        f"target S{target_sid}: acc={acc:.4f}, "
        f"itr={itr:.2f} bits/min "
        f"(avg over {len(source_sids)} sources)"
    )

    all_results.append(
        {
            "subject_id": target_sid,
            "subject": f"S{target_sid}",
            "acc": acc,
            "itr": itr,
        }
    )

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (IISTLF) ==========
n_classes=40, t_sec=1.5
target S1: acc=0.8469, itr=155.80 bits/min (avg over 69 sources)
target S2: acc=0.8905, itr=169.78 bits/min (avg over 69 sources)
target S3: acc=0.9033, itr=174.08 bits/min (avg over 69 sources)
target S4: acc=0.6575, itr=103.38 bits/min (avg over 69 sources)
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S31-S40.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S31-S40.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar

In [6]:
# **************************************************
# BETA数据集 eTRCA方法 - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== eTRCA 固定参数配置 ==========
srate = 250
T = 1.5         # 数据时长（秒）
n_harmonics = 3

# 滤波器组配置
wp = [(5, 90)]
ws = [(3, 92)]
filterbank = generate_filterbank(wp, ws, srate=srate, order=15, rp=0.5)

# 获取频率列表与类别数
events = BETA_dataset.events.keys()
freq_list = [BETA_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)   # 通常是40类

# 生成参考信号
Yf = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA reference shape: {Yf.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision window t_sec: {T}")

# ========== 一次性加载全体数据（不分组） ==========
all_subjects = list(range(1, 71))
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects ,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print("\n" + "#" * 70)
print("# 🔬 LOSO on ALL subjects")
print("#" * 70)
print(f"📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)

print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== 存储结果 ==========
acc_list = []
itr_list = []
subject_results = []

# ========== LOSO循环 ==========
for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分：当前被试测试，其余训练
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化SC_TRCA模型
    estimator = SC_TRCA(
        standard=False,   # 非标准TRCA
        ensemble=True,    # ensemble
        n_components=1
    )

    # 训练
    print("🔄 SC_TRCA训练中...")
    estimator.fit(X_train, y_train, Yf)
    print("✅ SC_TRCA训练完成")

    # 预测（SC_TRCA返回(相关系数, 标签)）
    _, p_labels = estimator.predict(X_test)

    # 指标计算
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总统计 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_subject = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_subject.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA reference shape: (40, 6, 375)
📊 Number of classes: 40
📊 Decision window t_sec: 1.5
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
------

In [7]:
# **************************************************
# BETA数据集 TtCCA - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== 固定参数 ==========
srate = 250
T = 1.5
n_harmonics = 3

events = BETA_dataset.events.keys()
freq_list = [BETA_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成CCA参考信号模板
cca_template = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA template shape: {cca_template.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")


# ========== 一次性加载全体数据（不分组） ==========
all_subjects = list(range(1, 71))
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)
print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化TtCCA模型
    estimator = TtCCA(
        n_components=1,
        n_jobs=-1
    )

    # 训练
    print("🔄 TtCCA训练中...")
    estimator.fit(X_train, y_train, cca_template, n_harmonics)
    print("✅ TtCCA训练完成")

    # 预测
    p_labels = estimator.predict(X_test)

    # 指标
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA template shape: (40, 6, 375)
📊 Number of classes: 40
📊 Decision time: 1.5s
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss

In [7]:
# TDCA
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec

# =========================
# TDCA参数区
# =========================
n_harmonics = 3
window_length = 1.5
ref_T = 1.4          # 避免TDCA长度冲突
padding_len = 1
n_components = 4

n_classes = len(freq_list)
t_sec = window_length

print(f"[BETA] n_classes={n_classes}, t_sec={t_sec:.2f}s")

# 参考信号（注意这里要float频率）
Yf = generate_cca_references(
    [float(f) for f in freq_list],
    srate=sfreq,
    T=ref_T,
    n_harmonics=n_harmonics
)
print(f"[BETA] CCA references shape: {Yf.shape}")

# =========================
# 一次性加载全体数据（LOSO）
# =========================
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects,
    return_concat=True,
    n_jobs=None,
    verbose=False
)

all_subject_ids = np.unique(meta_all["subject"])
N = len(all_subject_ids)

print("\n" + "#" * 70)
print("# 🔬 TDCA LOSO on BETA ALL subjects")
print("#" * 70)
print(f"Total subjects: {N}")
print(f"Subject IDs: {all_subject_ids}")
print(f"Data shape: {X_all.shape}")

# =========================
# LOSO循环
# =========================
acc_list = []
itr_list = []
results = []

for i, test_subject in enumerate(all_subject_ids):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all["subject"] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"Training samples: {len(train_ind)}")
    print(f"Testing samples: {len(test_ind)}")
    print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

    estimator = TDCA(
        padding_len=padding_len,
        n_components=n_components
    )

    # 安全检查，防止长度报错
    x_len = X_train.shape[-1]
    yf_len = Yf.shape[-1]
    if x_len <= (padding_len + yf_len):
        raise ValueError(
            f"Not enough time points for TDCA: "
            f"X_len={x_len}, padding_len={padding_len}, Yf_len={yf_len}, "
            f"need X_len > padding_len + Yf_len.\n"
            f"Try smaller ref_T, smaller padding_len, or longer data window."
        )

    p_labels = estimator.fit(
        X=X_train,
        y=y_train,
        Yf=Yf
    ).predict(X_test)

    acc_s = float(np.mean(p_labels == y_test))
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=t_sec))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    n_correct = int(np.sum(p_labels == y_test))
    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    results.append({
        "test_subject": int(test_subject),
        "accuracy": acc_s,
        "itr": itr_s,
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# =========================
# 汇总打印
# =========================
df_results = pd.DataFrame(results).sort_values("test_subject")

acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)

[BETA] n_classes=40, t_sec=1.50s
[BETA] CCA references shape: (40, 6, 350)
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upl

In [8]:
# EEGconformer
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（无目标校准、无目标微调）
# =========================
all_subjects = list(range(1, 71))
random_state = 42
n_classes = len(freq_list)
t_sec = 1.5

print("\n========== LOSO on ALL subjects (EEG-Conformer) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)
accs = []
itrs = []

for target_sid in all_subjects:
    X_test, y_test = all_data[target_sid]
    train_sids = [s for s in all_subjects if s != target_sid]
    X_train = np.concatenate([all_data[s][0] for s in train_sids], axis=0)
    y_train = np.concatenate([all_data[s][1] for s in train_sids], axis=0)

    model = EEGConformer(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        device="cuda" if torch.cuda.is_available() else "cpu",
        emb_size=40,
        depth=3,
        n_heads=4,
        batch_size=256,
        n_epochs=100,
        lr=3e-4,
        beta1=0.5,
        beta2=0.999,
        seed=random_state,
        use_amp=True,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test, batch_size=512)

    acc = float(np.mean(y_pred == y_test))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))
    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (EEG-Conformer) ==========
n_classes=40, t_sec=1.5
target S1: acc=0.5062, itr=68.49 bits/min
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S31-S40.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /uploa